# 주간 모델 학습 노트북

**매주 1회 돌리는 노트북**입니다. 위에서 아래로 순서대로 실행하면 됩니다.

### 전체 흐름

```
Step 0: 설정 (APP_NAME, Supabase/Server URL)
Step 1: 데이터 준비 — 3가지 소스에서 데이터 수집
  Step 1-1: Snowflake에서 features + outcomes 가져오기 (수동 CSV)
  Step 1-2: Supabase에서 prediction_logs 가져오기 (자동 다운로드)
  Step 1-3: Snowflake에서 modal_clicked 가져오기 (수동 CSV)
  Step 1-4: 3개 소스 JOIN → final weekly CSV 생성
Step 2: Feature 정의
Step 3: Factor Analysis
Step 4: D3 Purchase 모델 학습
Step 5: D3 Churn 모델 학습
Step 6: CATE 모델 학습 (is_random=1만)
Step 7: 최종 확인
Step 8: 테스트
Step 9: 서버 업로드
```

### 앱별 모델 분리
앱별로 모델이 다릅니다. `APP_NAME`을 바꾸면 다른 앱의 모델을 학습할 수 있어요.

### 데이터 소스 3가지

| 소스 | 가져오는 것 | 방법 |
|------|-----------|------|
| Snowflake (1-1) | UA features + In-app features + d3_purchase + d3_churn | SQL 실행 → CSV 다운로드 |
| Supabase (1-2) | assigned_trigger, is_random | Python 코드로 자동 다운로드 |
| Snowflake (1-3) | modal_clicked | SQL 실행 → CSV 다운로드 |

### 만들어지는 파일
| 파일 | 설명 | 업데이트 빈도 |
|------|------|-------------|
| `models/{APP_NAME}/fa_params.json` | FA 파라미터 (유저 성향 차원) | 거의 안 바뀜 |
| `models/{APP_NAME}/d3_purchase_model.pkl` | D3 구매 예측 | 매주 |
| `models/{APP_NAME}/d3_churn_model.pkl` | D3 이탈 예측 | 매주 |
| `models/{APP_NAME}/cate_model.pkl` | Trigger 반응 예측 (CATE) | 매주 |

### 학습 데이터 구조
- **전체 유저**: UA features + In-app features + d3_purchase + d3_churn
- **80% 모델 추천 유저**: `is_random=0` — 모델이 best trigger를 배정. 학습에 편향 있음.
- **20% 랜덤 유저**: `is_random=1` — 랜덤 배정. **CATE 학습에만 이 유저를 사용.**

---
## Step 0: 설정

앱 이름, 서버 URL, Supabase 설정을 입력합니다.

In [ ]:
import pandas as pd
import numpy as np
import glob
import json
import pickle
import os
from datetime import datetime, timedelta
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FactorAnalysis
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ===== 여기만 수정하세요 =====
APP_NAME = "ablog"
SERVER_URL = "https://airbridge-entry-api-prototype.onrender.com"

# Supabase (prediction_logs 다운로드용)
SUPABASE_URL = "https://nwujejkjdotdbfeygphc.supabase.co"
SUPABASE_KEY = ""  # ← Supabase anon key 입력 (또는 export SUPABASE_KEY=... 후 비워두기)
# ============================

# 환경변수 fallback
if not SUPABASE_KEY:
    SUPABASE_KEY = os.environ.get("SUPABASE_KEY", "")

today = datetime.now().strftime('%Y-%m-%d')
data_dir = f'../data/{APP_NAME}'
model_dir = f'../models/{APP_NAME}'
os.makedirs(data_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

# 데이터 기간 (최근 7일, D3 outcome 확정 위해 3일 전까지)
end_date = (datetime.now() - timedelta(days=3)).strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=10)).strftime('%Y-%m-%d')

print(f"앱: {APP_NAME}")
print(f"데이터 기간: {start_date} ~ {end_date}")
print(f"서버: {SERVER_URL}")
print(f"Supabase: {'설정됨' if SUPABASE_KEY else '미설정 (Step 1-2에서 CSV fallback 사용)'}")
print(f"오늘: {today}")

---
## Step 1: 데이터 준비

3가지 소스에서 데이터를 수집하고 JOIN합니다.

| Step | 소스 | 가져오는 것 | 방법 |
|------|------|-----------|------|
| 1-1 | Snowflake | features + d3_purchase + d3_churn | SQL 실행 → CSV 다운로드 |
| 1-2 | Supabase | assigned_trigger, is_random | Python 자동 다운로드 |
| 1-3 | Snowflake | modal_clicked, clicked_trigger | SQL 실행 → CSV 다운로드 |
| 1-4 | -- | 3개 JOIN → weekly CSV | 자동 |

### Step 1-1: Snowflake에서 features + outcomes 가져오기

Snowflake Web UI에서 아래 쿼리를 실행하고 CSV로 다운로드합니다.  
이 CSV에는 UA features, In-app features, d3_purchase, d3_churn이 포함됩니다.

**`modal_clicked`는 여기서 가져오지 않습니다** (Step 1-3에서 별도로 가져옴).  
**`assigned_trigger`, `is_random`도 여기서 가져오지 않습니다** (Step 1-2 Supabase에서 가져옴).

아래 셀을 실행하면 Snowflake 쿼리가 출력됩니다. 복사해서 Snowflake에서 실행하세요.

In [ ]:
# 앱별 config 로드 (이벤트 매핑 사용)
config_path = f'../configs/{APP_NAME}.json'
if os.path.exists(config_path):
    with open(config_path) as f:
        app_config = json.load(f)
    APP_ID = app_config.get('app_id', 'UNKNOWN')
    event_mapping = app_config.get('event_mapping', {})
    print(f"Config 로드: {config_path} (app_id={APP_ID})")
else:
    print(f"Config 없음: {config_path}")
    print(f"onboarding.ipynb를 먼저 실행하세요.")
    raise FileNotFoundError(f"{config_path} not found")

# Snowflake 쿼리에서 사용할 이벤트명
ev_product_viewed = event_mapping.get('product_viewed', 'airbridge.ecommerce.product.viewed')
ev_signin = event_mapping.get('signin', 'airbridge.user.signin')
ev_addedtocart = event_mapping.get('addedtocart', 'airbridge.ecommerce.product.addedToCart')
ev_home_viewed = event_mapping.get('home_viewed', 'airbridge.ecommerce.home.viewed')
ev_addtowishlist = event_mapping.get('addtowishlist', 'airbridge.addToWishlist')
ev_onboarding = event_mapping.get('onboarding', 'airbridge.onboarding')
ev_signup = event_mapping.get('signup', 'airbridge.user.signup')
ev_purchase = event_mapping.get('purchase', 'airbridge.ecommerce.order.completed')
ev_deeplink_cat = event_mapping.get('deeplink_open_category', '9162')

# leakage 이벤트 (outcome이므로 5분 feature에서 제외)
leakage_events = app_config.get('leakage_events', [ev_purchase, 'airbridge.initiateCheckout'])

sf_path = f'{data_dir}/snowflake_{today}.csv'

print(f"""
{'='*70}
Snowflake에서 아래 쿼리를 실행하고 CSV로 저장하세요.
파일명: {sf_path}
{'='*70}

-- ============================================================
-- Step 1: install_users (설치 유저 추출)
-- ============================================================
CREATE TEMPORARY TABLE install_users AS
SELECT
    DATA__DEVICE__AIRBRIDGEGENERATEDDEVICEUUID AS user_id,
    MIN(EVENT_TIMESTAMP) AS install_ts,
    MAX(DATA__DEVICE__OSNAME) AS OS_NAME,
    MAX(DATA__DEVICE__MANUFACTURER) AS DEVICE_MANUFACTURER,
    MAX(DATA__DEVICE__LANGUAGE) AS DEVICE_LANGUAGE,
    MAX(DATA__DEVICE__TIMEZONE) AS DEVICE_TIMEZONE,
    MAX(DATA__DEVICE__OSVERSION) AS DEVICE_OSVERSION,
    MAX(DATA__DEVICE__NETWORK__CARRIER) AS DEVICE_CARRIER
FROM AIRBRIDGE.PUBLIC.INTERNAL_MOBILE_EVENTS
WHERE DATA__APP__APPID = {APP_ID}
  AND DATA__EVENTDATA__CATEGORY = '9161'
  AND EVENT_TIMESTAMP >= '{start_date}'
  AND EVENT_TIMESTAMP < '{end_date}'
GROUP BY 1;

-- ============================================================
-- Step 2: fmt_flat (UA attribution 결과)
-- ============================================================
CREATE TEMPORARY TABLE fmt_flat AS
SELECT
    DATA__DEVICE:airbridgeGeneratedDeviceUUID::STRING AS user_id,
    DATA__ATTRIBUTIONRESULT AS journey_json,
    EVENT_TIMESTAMP AS fmt_ts
FROM AIRBRIDGE.PUBLIC.AIRBRIDGE_FMT_MOBILE_APP_EVENT_RESULTS_V20210802
WHERE APP_ID = {APP_ID}
  AND DATA__EVENTDATA__CATEGORY = '9161'
QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY fmt_ts DESC) = 1;

-- ============================================================
-- Step 3: ua_table (유저 + journey JOIN)
-- ============================================================
CREATE TEMPORARY TABLE ua_table AS
SELECT iu.*, ff.journey_json
FROM install_users iu
LEFT JOIN fmt_flat ff ON iu.user_id = ff.user_id;

-- ============================================================
-- Step 4: inapp_base (인앱 이벤트)
-- ============================================================
CREATE TEMPORARY TABLE inapp_base AS
SELECT
    ev.DATA__DEVICE__AIRBRIDGEGENERATEDDEVICEUUID AS user_id,
    ev.DATA__EVENTDATA__CATEGORY AS evt_cat,
    ev.DATA__EVENTDATA__GOAL__CATEGORY AS goal_cat,
    ev.EVENT_TIMESTAMP,
    TIMESTAMPDIFF(SECOND, iu.install_ts, ev.EVENT_TIMESTAMP) AS sec_after
FROM AIRBRIDGE.PUBLIC.INTERNAL_MOBILE_EVENTS ev
INNER JOIN install_users iu
    ON ev.DATA__DEVICE__AIRBRIDGEGENERATEDDEVICEUUID = iu.user_id
WHERE ev.DATA__APP__APPID = {APP_ID}
  AND ev.EVENT_TIMESTAMP >= iu.install_ts
  AND ev.EVENT_TIMESTAMP <= DATEADD(DAY, 30, iu.install_ts);

-- ============================================================
-- Step 5: user_churn_base (이탈 기준)
-- ============================================================
CREATE TEMPORARY TABLE user_churn_base AS
SELECT user_id, MAX(sec_after) AS seconds_active_after_install
FROM inapp_base GROUP BY 1;

-- ============================================================
-- Step 6: inapp_agg_5min (인앱 5분 집계 + d3 outcomes)
-- ============================================================
CREATE TEMPORARY TABLE inapp_agg_5min AS
SELECT
    user_id,
    COUNT(CASE WHEN goal_cat = '{ev_product_viewed}' AND sec_after <= 300 THEN 1 END) AS product_viewed_count,
    MAX(CASE WHEN goal_cat = '{ev_signin}' AND sec_after <= 300 THEN 1 ELSE 0 END) AS user_signin,
    MAX(CASE WHEN goal_cat = '{ev_addedtocart}' AND sec_after <= 300 THEN 1 ELSE 0 END) AS product_addedtocart,
    MAX(CASE WHEN evt_cat = '{ev_deeplink_cat}' AND sec_after <= 300 THEN 1 ELSE 0 END) AS deeplink_open,
    MAX(CASE WHEN goal_cat = '{ev_home_viewed}' AND sec_after <= 300 THEN 1 ELSE 0 END) AS home_viewed,
    MAX(CASE WHEN goal_cat = '{ev_addtowishlist}' AND sec_after <= 300 THEN 1 ELSE 0 END) AS addtowishlist,
    MAX(CASE WHEN goal_cat = '{ev_onboarding}' AND sec_after <= 300 THEN 1 ELSE 0 END) AS onboarding,
    MAX(CASE WHEN goal_cat = '{ev_signup}' AND sec_after <= 300 THEN 1 ELSE 0 END) AS user_signup,
    COUNT(CASE WHEN sec_after <= 300 AND evt_cat != '9161'
               AND COALESCE(goal_cat, '') NOT IN ('{leakage_events[0]}', '{leakage_events[1] if len(leakage_events) > 1 else ""}')
               THEN 1 END) AS total_events,
    COUNT(DISTINCT CASE WHEN sec_after <= 300 AND evt_cat != '9161'
               AND COALESCE(goal_cat, '') NOT IN ('{leakage_events[0]}', '{leakage_events[1] if len(leakage_events) > 1 else ""}')
               THEN COALESCE(goal_cat, evt_cat) END) AS n_event_types,
    MAX(CASE WHEN goal_cat = '{ev_purchase}' AND sec_after <= 259200 THEN 1 ELSE 0 END) AS d3_purchase
FROM inapp_base GROUP BY 1;

-- ============================================================
-- Step 12-13: Facebook 터치포인트 복원 (선택)
-- ============================================================
CREATE TEMPORARY TABLE ua_table_enriched AS
SELECT ut.*, rf.meta_touchpoint
FROM ua_table ut
LEFT JOIN (
    SELECT ru.user_id, fb.DATA AS meta_touchpoint
    FROM ua_table ru
    INNER JOIN AB180_X_FACEBOOK.PUBLIC.AIRBRIDGE_PLATFORM_MATCHED_TOUCHPOINTS_V20230410 fb
        ON fb.APP_ID = {APP_ID}
        AND fb.CONVERSION_EVENT_TIMESTAMP = ru.install_ts
    WHERE ru.journey_json LIKE '%"restricted"%'
) rf ON ut.user_id = rf.user_id;

-- ============================================================
-- 최종 결합 쿼리 (이 결과를 CSV로 다운로드)
-- ============================================================
SELECT
    {APP_ID} AS app_id,
    ue.user_id AS airbridge_uuid,
    ue.install_ts,
    ue.OS_NAME, ue.DEVICE_MANUFACTURER, ue.DEVICE_LANGUAGE,
    ue.DEVICE_TIMEZONE, ue.DEVICE_OSVERSION, ue.DEVICE_CARRIER,
    ue.journey_json, ue.meta_touchpoint,
    COALESCE(ia.product_viewed_count, 0) AS product_viewed_count,
    COALESCE(ia.user_signin, 0) AS user_signin,
    COALESCE(ia.product_addedtocart, 0) AS product_addedtocart,
    COALESCE(ia.deeplink_open, 0) AS deeplink_open,
    COALESCE(ia.home_viewed, 0) AS home_viewed,
    COALESCE(ia.addtowishlist, 0) AS addtowishlist,
    COALESCE(ia.onboarding, 0) AS onboarding,
    COALESCE(ia.user_signup, 0) AS user_signup,
    COALESCE(ia.total_events, 0) AS total_events,
    COALESCE(ia.n_event_types, 0) AS n_event_types,
    COALESCE(ia.d3_purchase, 0) AS d3_purchase,
    CASE WHEN COALESCE(uc.seconds_active_after_install, 0) <= 259200 THEN 1 ELSE 0 END AS d3_churn,
    COALESCE(ta.IS_HAS_FRAUD, 0) AS IS_HAS_FRAUD,
    ta.sa_term_list
FROM ua_table_enriched ue
LEFT JOIN inapp_agg_5min ia ON ue.user_id = ia.user_id
LEFT JOIN user_churn_base uc ON ue.user_id = uc.user_id
LEFT JOIN touchpoint_analysis ta ON ue.user_id = ta.user_id;
""")

print(f"CSV를 {sf_path} 에 저장한 후 다음 셀을 실행하세요.")

In [ ]:
# Snowflake CSV 로드
sf_path = f'{data_dir}/snowflake_{today}.csv'

if os.path.exists(sf_path):
    df_sf = pd.read_csv(sf_path)
    print(f"Snowflake CSV 로드: {sf_path} ({len(df_sf)}명)")
    print(f"  컬럼: {list(df_sf.columns)[:10]}...")
else:
    # 다른 날짜 파일이 있는지 확인
    sf_files = sorted(glob.glob(f'{data_dir}/snowflake_*.csv'))
    if sf_files:
        sf_path = sf_files[-1]
        df_sf = pd.read_csv(sf_path)
        print(f"오늘 날짜 파일 없음. 최신 파일 사용: {sf_path} ({len(df_sf)}명)")
    else:
        print(f"파일 없음: {sf_path}")
        print(f"위 셀의 Snowflake 쿼리를 실행하고 CSV로 저장하세요.")
        df_sf = None

if df_sf is None:
    raise FileNotFoundError(f"Snowflake CSV가 없습니다. {data_dir}/snowflake_*.csv")

### Step 1-2: Supabase에서 prediction_logs 가져오기

서버가 API 호출 시마다 Supabase에 기록한 로그입니다.  
각 유저에게 **어떤 trigger가 배정되었는지** (`assigned_trigger`)와 **랜덤 배정 여부** (`is_random`)를 가져옵니다.

- `SUPABASE_KEY`가 설정되어 있으면 자동으로 다운로드합니다.
- 설정되어 있지 않으면 CSV fallback을 사용합니다.

In [ ]:
df_logs = None

if SUPABASE_KEY:
    try:
        from supabase import create_client
        client = create_client(SUPABASE_URL, SUPABASE_KEY)

        # prediction_logs 전체 다운로드 (이 앱만)
        result = client.table("prediction_logs") \
            .select("user_id, best_trigger, is_random, timestamp") \
            .eq("app_id", APP_NAME) \
            .execute()

        if result.data:
            df_logs = pd.DataFrame(result.data)
            df_logs = df_logs.rename(columns={
                "user_id": "airbridge_uuid",
                "best_trigger": "assigned_trigger"
            })
            # is_random 타입 통일 (bool -> int)
            df_logs['is_random'] = df_logs['is_random'].astype(int)

            # 중복 제거 (같은 유저가 여러번 호출된 경우 최신 것만)
            df_logs = df_logs.sort_values('timestamp').drop_duplicates(
                subset=['airbridge_uuid'], keep='last'
            )
            df_logs = df_logs[['airbridge_uuid', 'assigned_trigger', 'is_random']]

            print(f"Supabase에서 {len(df_logs)}건 로드")
            print(f"  랜덤 배정: {(df_logs['is_random']==1).sum()}명")
            print(f"  모델 추천: {(df_logs['is_random']==0).sum()}명")
            print(f"  Trigger 분포:")
            print(df_logs['assigned_trigger'].value_counts().to_string(header=False))

            # CSV로도 백업 저장
            backup_path = f'{data_dir}/prediction_logs_{today}.csv'
            df_logs.to_csv(backup_path, index=False)
            print(f"\n  백업 저장: {backup_path}")
        else:
            print("Supabase에 이 앱의 prediction_logs가 없습니다.")
            print("서버가 아직 API 호출을 받지 않았을 수 있습니다.")
    except ImportError:
        print("supabase 패키지가 없습니다. 설치하세요: pip install supabase")
    except Exception as e:
        print(f"Supabase 연결 실패: {e}")
else:
    print("SUPABASE_KEY가 설정되지 않았습니다.")
    print("방법 1: Step 0에서 SUPABASE_KEY를 직접 입력")
    print("방법 2: export SUPABASE_KEY='your-key-here' 후 노트북 재시작")

# CSV fallback
if df_logs is None:
    csv_fallback = sorted(glob.glob(f'{data_dir}/prediction_logs*.csv'))
    if csv_fallback:
        fallback_path = csv_fallback[-1]
        df_logs = pd.read_csv(fallback_path)
        print(f"\nCSV fallback 사용: {fallback_path} ({len(df_logs)}건)")
    else:
        print(f"\nCSV fallback도 없습니다: {data_dir}/prediction_logs*.csv")
        print("assigned_trigger, is_random 없이 진행합니다.")
        print("(D3 Purchase/Churn은 학습 가능, CATE는 학습 불가)")

### Step 1-3: Snowflake에서 modal_clicked 가져오기

`entry_modal_clicked` 이벤트를 Snowflake에서 별도로 추출합니다.  
이 이벤트는 고객사가 SDK로 태깅한 커스텀 이벤트입니다.

아래 셀을 실행하면 쿼리가 출력됩니다. Snowflake에서 실행 후 CSV로 저장하세요.

In [ ]:
ev_modal_clicked = event_mapping.get('modal_clicked', 'entry_modal_clicked')
outcomes_path = f'{data_dir}/outcomes_{today}.csv'

print(f"""
{'='*70}
Snowflake에서 아래 쿼리를 실행하고 CSV로 저장하세요.
파일명: {outcomes_path}
{'='*70}

SELECT
    DATA__DEVICE__AIRBRIDGEGENERATEDDEVICEUUID AS airbridge_uuid,
    1 AS modal_clicked,
    PARSE_JSON(DATA__EVENTDATA__GOAL__CUSTOMATTRIBUTES):trigger_type::STRING AS clicked_trigger
FROM AIRBRIDGE.PUBLIC.INTERNAL_MOBILE_EVENTS
WHERE DATA__APP__APPID = {APP_ID}
  AND DATA__EVENTDATA__GOAL__CATEGORY = '{ev_modal_clicked}'
  AND EVENT_TIMESTAMP >= '{start_date}'
  AND EVENT_TIMESTAMP < DATEADD('day', 7, '{end_date}')
GROUP BY 1, 2, 3;
""")

print(f"CSV를 {outcomes_path} 에 저장한 후 다음 셀을 실행하세요.")

In [ ]:
# Outcomes CSV 로드
outcomes_path = f'{data_dir}/outcomes_{today}.csv'
df_outcomes = None

if os.path.exists(outcomes_path):
    df_outcomes = pd.read_csv(outcomes_path)
    print(f"Outcomes CSV 로드: {outcomes_path} ({len(df_outcomes)}건)")
else:
    # 다른 날짜 파일이 있는지 확인
    outcome_files = sorted(glob.glob(f'{data_dir}/outcomes_*.csv'))
    if outcome_files:
        outcomes_path = outcome_files[-1]
        df_outcomes = pd.read_csv(outcomes_path)
        print(f"오늘 날짜 파일 없음. 최신 파일 사용: {outcomes_path} ({len(df_outcomes)}건)")
    else:
        print(f"파일 없음: {outcomes_path}")
        print(f"위 셀의 Snowflake 쿼리를 실행하고 CSV로 저장하세요.")
        print(f"(또는 entry_modal_clicked 이벤트가 아직 태깅되지 않았을 수 있습니다)")
        print(f"modal_clicked=0으로 진행합니다.")

### Step 1-4: 3개 소스 JOIN → weekly CSV 생성

Snowflake features + Supabase prediction_logs + Snowflake modal_clicked를 `airbridge_uuid`로 JOIN합니다.

- Supabase에 없는 유저 → `assigned_trigger=NaN`, `is_random=NaN` (API 호출 전 유저)
- modal_clicked 이벤트 없는 유저 → `modal_clicked=0`

In [ ]:
# === 3개 소스 JOIN ===

df = df_sf.copy()
print(f"[1] Snowflake features: {len(df)}명")

# JOIN 1: Supabase prediction_logs (assigned_trigger, is_random)
if df_logs is not None and len(df_logs) > 0:
    df = df.merge(
        df_logs[['airbridge_uuid', 'assigned_trigger', 'is_random']],
        on='airbridge_uuid', how='left'
    )
    matched = df['assigned_trigger'].notna().sum()
    print(f"[2] + Supabase prediction_logs: {matched}/{len(df)}명 매칭 ({matched/len(df):.1%})")
else:
    df['assigned_trigger'] = None
    df['is_random'] = None
    print(f"[2] Supabase prediction_logs 없음 → assigned_trigger=NaN, is_random=NaN")

# JOIN 2: Snowflake modal_clicked
if df_outcomes is not None and len(df_outcomes) > 0:
    # airbridge_uuid 컬럼명 통일 (대소문자 차이 가능)
    if 'AIRBRIDGE_UUID' in df_outcomes.columns:
        df_outcomes = df_outcomes.rename(columns={'AIRBRIDGE_UUID': 'airbridge_uuid'})

    df = df.merge(
        df_outcomes[['airbridge_uuid', 'modal_clicked']].drop_duplicates(subset=['airbridge_uuid']),
        on='airbridge_uuid', how='left'
    )
    df['modal_clicked'] = df['modal_clicked'].fillna(0).astype(int)
    clicked = df['modal_clicked'].sum()
    print(f"[3] + Snowflake modal_clicked: {clicked}명 클릭 ({clicked/len(df):.1%})")
else:
    df['modal_clicked'] = 0
    print(f"[3] Snowflake modal_clicked 없음 → modal_clicked=0")

# 저장
save_path = f'{data_dir}/weekly_{today}.csv'
df.to_csv(save_path, index=False)
print(f"\n{'='*50}")
print(f"weekly CSV 저장: {save_path} ({len(df)}명)")
print(f"{'='*50}")

# 요약 통계
print(f"\n=== 데이터 요약 ===")
print(f"총 유저 수: {len(df)}")
if df['is_random'].notna().any():
    random_count = (df['is_random'] == 1).sum()
    model_count = (df['is_random'] == 0).sum()
    na_count = df['is_random'].isna().sum()
    print(f"랜덤 배정 (is_random=1): {random_count}명 ({random_count/len(df):.1%})")
    print(f"모델 추천 (is_random=0): {model_count}명 ({model_count/len(df):.1%})")
    if na_count > 0:
        print(f"API 미호출 (is_random=NaN): {na_count}명 ({na_count/len(df):.1%})")
print(f"D3 구매율: {df['d3_purchase'].mean():.1%}")
print(f"D3 이탈율: {df['d3_churn'].mean():.1%}")
print(f"모달 클릭률: {df['modal_clicked'].mean():.1%}")

if df['assigned_trigger'].notna().any():
    print(f"\nTrigger 배정:")
    print(df['assigned_trigger'].value_counts().to_string())
    if df['modal_clicked'].sum() > 0 and df['is_random'].notna().any():
        random_ctr = df[df['is_random']==1]['modal_clicked'].mean() if (df['is_random']==1).any() else 0
        model_ctr = df[df['is_random']==0]['modal_clicked'].mean() if (df['is_random']==0).any() else 0
        print(f"\n모달 클릭률:")
        print(f"  랜덤 유저: {random_ctr:.1%}")
        print(f"  모델 유저: {model_ctr:.1%}")
        if model_ctr > random_ctr:
            print(f"  → 모델 유저가 더 높음 = 모델이 잘 작동하는 것!")

---
## Step 2: Feature 정의

서비스에서 사용하는 feature는 총 25개입니다.

- **UA features (15개)**: 앱 오픈 전 광고 여정 데이터. Airbridge DB에서 조회.
- **In-app features (10개)**: 앱 오픈 후 첫 5분 행동 데이터. SDK에서 자동 수집.

D3 Purchase, D3 Churn 모델은 25개 전부 사용.  
CATE 모델도 25개 전부 사용 (trigger 반응 예측).

In [2]:
# UA features (15개) — 광고 여정
UA_FEATURES = [
    'trackinglink_count', 'DA_count', 'SA_count',
    'unique_channel_count', 'channel_entropy', 'last_touch_is_da',
    'latency', 'recency', 'recent_touch_pressure',
    'touch_per_latency_hour', 'last1h_touch_count', 'recent_24h_ratio',
    'click_ratio', 'impression_count', 'is_single_touch_install'
]

# In-app features (10개) — 첫 5분 행동
INAPP_FEATURES = [
    'product_viewed_count', 'user_signin', 'product_addedtocart',
    'deeplink_open', 'home_viewed', 'addtowishlist',
    'onboarding', 'user_signup', 'total_events', 'n_event_types'
]

ALL_FEATURES = UA_FEATURES + INAPP_FEATURES
TRIGGERS = ['price_appeal', 'social_proof', 'scarcity', 'novelty']

print(f'UA: {len(UA_FEATURES)}개 / In-app: {len(INAPP_FEATURES)}개 / 전체: {len(ALL_FEATURES)}개')
print(f'Trigger 종류: {TRIGGERS}')

UA: 15개 / In-app: 10개 / 전체: 25개
Trigger 종류: ['price_appeal', 'social_proof', 'scarcity', 'novelty']


---
## Step 3: Factor Analysis — UA 15개 → 6개 잠재 차원

UA 15개 feature를 6개 핵심 차원으로 압축합니다.  
"이 유저의 광고 여정이 어떤 성격인지"를 요약하는 거예요.

6개 차원: 광고 강도, DA 채널, 수동 노출, 채널 다양성, 시간 집중도, 최근성

**이건 연구용/API 응답용이고, 예측 모델에는 원본 25개 feature를 그대로 씁니다.**  
파라미터가 거의 안 바뀌어서 매주 돌리긴 하지만 실질적 변화는 거의 없음.

In [ ]:
# Organic 유저 제외 (UA가 전부 0이라 FA에 넣으면 안 됨)
paid_mask = df['trackinglink_count'] > 0
X_ua = df.loc[paid_mask, UA_FEATURES].values

# 정규화 (z-score)
scaler = StandardScaler()
X_ua_scaled = scaler.fit_transform(X_ua)

# Factor Analysis (6개 차원 추출)
fa = FactorAnalysis(n_components=6, rotation='varimax', random_state=42)
fa.fit(X_ua_scaled)

loading_matrix = fa.components_.T  # (15, 6)

factor_names = [
    'ad_intensity',       # 광고 강도
    'display_ad',         # DA 채널
    'passive_exposure',   # 수동 노출
    'channel_diversity',  # 채널 다양성
    'time_concentration', # 시간 집중도
    'recency'             # 최근성
]

# Loading matrix 출력
print('Loading Matrix (어떤 feature가 어떤 차원에 기여하는지):')
print(pd.DataFrame(loading_matrix, index=UA_FEATURES, columns=factor_names).round(2))

# 파라미터 저장
fa_params = {
    'mean': scaler.mean_.tolist(),
    'std': scaler.scale_.tolist(),
    'loading_matrix': loading_matrix.tolist(),
    'feature_names': UA_FEATURES,
    'factor_names': factor_names
}

with open(f'../models/{APP_NAME}/fa_params.json', 'w') as f:
    json.dump(fa_params, f, indent=2)

print(f'\n../models/{APP_NAME}/fa_params.json 저장 완료!')
print(f'  서버에서 하는 계산: (UA 15개 - mean) / std @ loading_matrix = 6개 스코어')

---
## Step 4: D3 Purchase 모델

25개 feature로 "3일 내 구매 여부"를 예측합니다.

**전체 유저(1000명)로 학습합니다.** 구매 예측은 trigger 배정과 무관하니까  
is_random=0인 유저도 is_random=1인 유저도 전부 쓸 수 있음.

In [ ]:
# 전체 유저로 학습
X_all = df[ALL_FEATURES].values
y_purchase = df['d3_purchase'].values

purchase_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
purchase_model.fit(X_all, y_purchase)

# 성능 확인 (5-fold CV)
scores_purchase = cross_val_score(purchase_model, X_all, y_purchase, cv=5, scoring='roc_auc')
print(f'D3 Purchase AUC: {scores_purchase.mean():.3f} (±{scores_purchase.std():.3f})')
print(f'  → 0.5=동전던지기, 0.8+=실무에서 쓸만함')

# Top/Bottom 비교
pred_proba = purchase_model.predict_proba(X_all)[:, 1]
df_eval = pd.DataFrame({'pred': pred_proba, 'actual': y_purchase})
df_eval['group'] = pd.qcut(df_eval['pred'], 5, labels=['하위20%', '하위40%', '중간', '상위40%', '상위20%'], duplicates='drop')

print(f'\n=== 예측 등급별 실제 구매율 ===')
result = df_eval.groupby('group')['actual'].agg(['mean', 'count'])
result.columns = ['실제 구매율', '유저 수']
for idx, row in result.iterrows():
    print(f'  {idx}: {row["실제 구매율"]:.1%} (n={row["유저 수"]:.0f})')

top = df_eval[df_eval['group'] == '상위20%']['actual'].mean()
bot = df_eval[df_eval['group'] == '하위20%']['actual'].mean()
print(f'\n  상위 20%: {top:.1%} / 하위 20%: {bot:.1%} / 비율: {top/max(bot,0.001):.1f}x')

# 저장
purchase_data = {'model': purchase_model, 'feature_names': ALL_FEATURES}
with open(f'../models/{APP_NAME}/d3_purchase_model.pkl', 'wb') as f:
    pickle.dump(purchase_data, f)
print(f'\n../models/{APP_NAME}/d3_purchase_model.pkl 저장 완료!')

---
## Step 5: D3 Churn 모델

동일한 25개 feature로 "3일 내 이탈 여부"를 예측합니다.  
구매 모델과 구조가 완전히 같고 label만 다릅니다. 역시 **전체 유저**로 학습.

In [ ]:
y_churn = df['d3_churn'].values

churn_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
churn_model.fit(X_all, y_churn)

# 성능 확인
scores_churn = cross_val_score(churn_model, X_all, y_churn, cv=5, scoring='roc_auc')
print(f'D3 Churn AUC: {scores_churn.mean():.3f} (±{scores_churn.std():.3f})')

# Top/Bottom 비교
pred_churn = churn_model.predict_proba(X_all)[:, 1]
df_eval_c = pd.DataFrame({'pred': pred_churn, 'actual': y_churn})
df_eval_c['group'] = pd.qcut(df_eval_c['pred'], 5, labels=['하위20%', '하위40%', '중간', '상위40%', '상위20%'], duplicates='drop')

print(f'\n=== 예측 등급별 실제 이탈율 ===')
result_c = df_eval_c.groupby('group')['actual'].agg(['mean', 'count'])
result_c.columns = ['실제 이탈율', '유저 수']
for idx, row in result_c.iterrows():
    print(f'  {idx}: {row["실제 이탈율"]:.1%} (n={row["유저 수"]:.0f})')

# 저장
churn_data = {'model': churn_model, 'feature_names': ALL_FEATURES}
with open(f'../models/{APP_NAME}/d3_churn_model.pkl', 'wb') as f:
    pickle.dump(churn_data, f)
print(f'\n../models/{APP_NAME}/d3_churn_model.pkl 저장 완료!')

---
## Step 6: CATE 모델 (Trigger 반응 예측) — S-Learner

**핵심 포인트: `is_random=1`인 유저만 사용합니다.**

왜? 80% 모델 추천 유저는 이미 "잘 맞을 것 같은 trigger"를 받았기 때문에 편향되어 있음.  
이 유저들로 학습하면 Simpson's Paradox로 인과 추정이 왜곡됩니다.

20% 랜덤 유저만이 진짜 RCT 데이터이고, CATE 학습에 쓸 수 있음.

**S-Learner 방식**: 하나의 모델에 trigger를 one-hot feature로 포함하여 학습.  
새 유저가 오면 4개 trigger one-hot을 바꿔가며 예측, 가장 클릭 확률이 높은 trigger를 추천.

T-Learner(trigger별 별도 모델) 대비 장점:
- 모든 데이터를 하나의 모델이 공유 → 소규모 데이터에서 안정적
- trigger 간 feature 효과를 공유할 수 있음

In [ ]:
# CATE 학습: 반드시 랜덤 배정 유저만 사용! (is_random=1)
# 80% 모델 추천 유저를 포함하면 Simpson's Paradox로 인과 추정이 왜곡됩니다.

# is_random이 없거나 전부 NaN인 경우 체크
has_random_data = df['is_random'].notna().any() and (df['is_random'] == 1).any()
has_trigger_data = df['assigned_trigger'].notna().any()
has_click_data = df['modal_clicked'].notna().any() and df['modal_clicked'].sum() > 0

if not has_random_data:
    print("is_random=1인 유저가 없습니다.")
    print("Supabase prediction_logs가 없거나, 아직 랜덤 배정 유저가 부족합니다.")
    print("CATE 학습을 건너뜁니다. (exploration 모드 유지)")
    cate_model = None
    df_cate = pd.DataFrame()
elif not has_trigger_data:
    print("assigned_trigger가 없습니다. CATE 학습을 건너뜁니다.")
    cate_model = None
    df_cate = pd.DataFrame()
elif not has_click_data:
    print("modal_clicked가 모두 0입니다. CATE를 학습해도 의미가 없습니다.")
    print("entry_modal_clicked 이벤트가 태깅되었는지 확인하세요.")
    cate_model = None
    df_cate = pd.DataFrame()
else:
    df_cate = df[(df['is_random'] == 1) & df['assigned_trigger'].notna()].copy()
    print(f'CATE 학습: 랜덤 유저 {len(df_cate)}명만 사용 (모델 추천 유저 {len(df) - len(df_cate)}명 제외)')

    if len(df_cate) < 50:
        print(f"\n랜덤 유저가 {len(df_cate)}명으로 너무 적습니다. (최소 50명 권장)")
        print("1-2주 더 데이터를 모은 후 다시 시도하세요.")
        cate_model = None
    else:
        # S-Learner: 하나의 모델, trigger를 one-hot feature로 포함
        X_rows = []
        y_rows = []
        for _, row in df_cate.iterrows():
            features = row[ALL_FEATURES].values.astype(float)
            trigger = row['assigned_trigger']
            trigger_onehot = [1 if t == trigger else 0 for t in TRIGGERS]
            X_rows.append(np.concatenate([features, trigger_onehot]))
            y_rows.append(row['modal_clicked'])

        X_cate = np.array(X_rows)
        y_cate = np.array(y_rows)

        print(f'S-Learner 학습 데이터: {X_cate.shape[0]}행 x {X_cate.shape[1]}열 (25 features + 4 trigger one-hot)')
        print()

        # 단일 모델 학습
        cate_model = RandomForestClassifier(
            n_estimators=200, max_depth=8, min_samples_leaf=10, random_state=42
        )
        cate_model.fit(X_cate, y_cate)

        # ATE 분석 (랜덤 유저 기준)
        print(f'=== ATE 분석 (랜덤 유저 기준) ===')
        overall_rate = df_cate['modal_clicked'].mean()
        print(f'전체 클릭률: {overall_rate:.1%}')
        print()
        print(f'{"Trigger":15s} | {"클릭률":>8s} | {"vs 평균":>8s} | {"p-value":>10s}')
        print('-' * 55)

        for trigger in TRIGGERS:
            mask = df_cate['assigned_trigger'] == trigger
            if mask.sum() == 0:
                print(f'{trigger:15s} | {"N/A":>8s} | {"N/A":>8s} | {"N/A":>10s}')
                continue
            treat_rate = df_cate.loc[mask, 'modal_clicked'].mean()
            others = df_cate.loc[~mask, 'modal_clicked']
            treat = df_cate.loc[mask, 'modal_clicked']
            if len(treat) > 1 and len(others) > 1:
                _, p_val = stats.ttest_ind(treat, others)
                diff = treat_rate - overall_rate
                print(f'{trigger:15s} | {treat_rate:>7.1%} | {diff:>+7.1%} | {p_val:>10.4f}')
            else:
                diff = treat_rate - overall_rate
                print(f'{trigger:15s} | {treat_rate:>7.1%} | {diff:>+7.1%} | {"N/A":>10s}')

In [ ]:
# CATE 모델 저장
if cate_model is not None:
    cate_data = {
        'model': cate_model,
        'treatment_triggers': TRIGGERS,
        'feature_names': ALL_FEATURES,
    }

    with open(f'../models/{APP_NAME}/cate_model.pkl', 'wb') as f:
        pickle.dump(cate_data, f)

    size = os.path.getsize(f'../models/{APP_NAME}/cate_model.pkl') / 1024 / 1024
    print(f'../models/{APP_NAME}/cate_model.pkl 저장 완료! ({size:.1f} MB)')
    print(f'  S-Learner: 단일 모델 (trigger를 one-hot feature로 포함)')
    print(f'  학습 데이터: 랜덤 유저 {len(df_cate)}명만 사용')
else:
    print("CATE 모델이 학습되지 않았습니다. (exploration 모드 유지)")
    cate_path = f'../models/{APP_NAME}/cate_model.pkl'
    if os.path.exists(cate_path):
        print(f"  기존 CATE 모델이 있습니다: {cate_path}")
        print(f"  이번 주에는 업데이트하지 않습니다.")

---
## Step 7: 최종 확인

서버에 업로드할 파일 4개를 확인합니다.

In [ ]:
print(f'=== 서버에 업로드할 파일 ({APP_NAME}) ===')
print()

files = [
    ('fa_params.json', 'FA 파라미터 (유저 성향 차원, 거의 안 바뀜)'),
    ('d3_purchase_model.pkl', f'D3 구매 예측 (AUC={scores_purchase.mean():.3f}, 전체 {len(df)}명으로 학습)'),
    ('d3_churn_model.pkl', f'D3 이탈 예측 (AUC={scores_churn.mean():.3f}, 전체 {len(df)}명으로 학습)'),
    ('cate_model.pkl', f'CATE 모델 ({"랜덤 유저 " + str(len(df_cate)) + "명으로 학습" if cate_model is not None else "미학습 — exploration 모드"})'),
]

for fname, desc in files:
    path = f'../models/{APP_NAME}/{fname}'
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        unit = 'KB'
        if size > 1024:
            size /= 1024
            unit = 'MB'
        print(f'  [OK] {fname:30s} ({size:.1f} {unit}) — {desc}')
    else:
        print(f'  [--] {fname:30s} — {desc}')

---
## Step 8: 테스트 — 유저 1명 예측

서버에서 API 호출이 들어왔을 때 일어나는 일을 그대로 재현합니다.  
3개 모델 전부 사용해서 완전한 API 응답을 만들어봅니다.

In [ ]:
# 유저 1명 선택 (paid 유저)
user = df[df['trackinglink_count'] > 0].iloc[0]
print(f'===== 유저 {user["airbridge_uuid"]} — 전체 예측 결과 =====')
print()

# 1. Latent dimensions (FA)
ua_raw = user[UA_FEATURES].values.astype(float)
ua_norm = (ua_raw - np.array(fa_params['mean'])) / np.array(fa_params['std'])
latent = ua_norm @ np.array(fa_params['loading_matrix'])

print('1) Latent Dimensions (광고 여정 성향):')
latent_dict = {}
for name, val in zip(factor_names, latent):
    latent_dict[name] = round(float(val), 4)
    print(f'   {name}: {val:.2f}')

# 2. D3 Purchase / Churn
features = user[ALL_FEATURES].values.astype(float).reshape(1, -1)
purchase_prob = purchase_model.predict_proba(features)[0][1]
churn_prob = churn_model.predict_proba(features)[0][1]

print(f'\n2) D3 예측:')
print(f'   구매 확률: {purchase_prob:.2f}')
print(f'   이탈 확률: {churn_prob:.2f}')

# 3. CATE — Trigger scores (S-Learner)
if cate_model is not None:
    print(f'\n3) Trigger Scores (S-Learner CATE):')
    user_features = user[ALL_FEATURES].values.astype(float)
    trigger_scores = {}
    for trigger in TRIGGERS:
        trigger_onehot = [1 if t == trigger else 0 for t in TRIGGERS]
        x = np.concatenate([user_features, trigger_onehot]).reshape(1, -1)
        prob = cate_model.predict_proba(x)[0][1]
        trigger_scores[trigger] = round(float(prob), 4)
        print(f'   {trigger:15s}: {prob:.4f}')

    best_trigger = max(trigger_scores, key=trigger_scores.get)
    print(f'   → best_trigger: {best_trigger}')
else:
    print(f'\n3) CATE 모델 없음 → trigger는 랜덤 배정 (exploration 모드)')
    trigger_scores = None
    best_trigger = "random"

# 4. 최종 API 응답
print(f'\n===== API 응답 =====')
response = {
    'user_id': user['airbridge_uuid'],
    'best_trigger': best_trigger,
    'trigger_scores': trigger_scores,
    'd3_purchase_prob': round(float(purchase_prob), 4),
    'd3_churn_prob': round(float(churn_prob), 4),
}
print(json.dumps(response, indent=2, ensure_ascii=False))

---
## Step 9: 서버에 모델 업로드

학습한 모델 파일을 서버에 API로 업로드합니다.  
서버가 자동으로 모델을 리로드하므로 재배포 없이 바로 반영됩니다.

**SERVER_URL을 실제 서버 주소로 바꿔주세요.**

In [ ]:
import requests

# SERVER_URL은 Step 0에서 설정됨
# SERVER_URL = "http://localhost:8000"  # 로컬 테스트 시 주석 해제

# 업로드할 파일 목록
model_files = [
    f"../models/{APP_NAME}/fa_params.json",
    f"../models/{APP_NAME}/d3_purchase_model.pkl",
    f"../models/{APP_NAME}/d3_churn_model.pkl",
    f"../models/{APP_NAME}/cate_model.pkl",
]

print(f"=== 서버에 모델 업로드 ({APP_NAME}) ===")
print(f"서버: {SERVER_URL}")
print()

for filepath in model_files:
    filename = os.path.basename(filepath)

    if not os.path.exists(filepath):
        print(f"  [SKIP] {filename} — 파일 없음")
        continue

    size_kb = os.path.getsize(filepath) / 1024

    try:
        with open(filepath, "rb") as f:
            resp = requests.post(
                f"{SERVER_URL}/v1/models/{APP_NAME}/upload",
                files={"file": (filename, f)},
            )

        if resp.status_code == 200:
            result = resp.json()
            print(f"  [OK] {filename:30s} ({size_kb:.1f} KB) → {result['mode']}")
        else:
            print(f"  [ERROR] {filename}: {resp.status_code} {resp.text}")
    except requests.ConnectionError:
        print(f"  [ERROR] 서버에 연결할 수 없습니다. SERVER_URL을 확인하세요: {SERVER_URL}")
        break

# 업로드 후 서버 상태 확인
print()
try:
    health = requests.get(f"{SERVER_URL}/health").json()
    print(f"서버 상태: {health['loaded_apps']}")
except:
    print("서버 상태 확인 실패")